# Ranking de cenários (Entrega 2)

Reúne os 10 melhores cenários da planilha da Entrega 1 de Contabilidade

- **Entrada:** `src/Entrega 2/Staging/kpis.parquet`, gerado por `kpis.ipynb`.
- **Saídas:** `src/Entrega 2/Staging/top10_referencia.csv` (blocos 1 e 2) e `src/Entrega 2/Staging/top10_ranking.csv`


## Bloco 1 - Top 10 da planilha

Posição, cenário e score final dos 10 melhores cenários, copiados da aba `ranking_cenarios`.

In [ ]:
from pathlib import Path

import pandas as pd

PASTA_STAGING = Path("../Staging")

top10 = pd.DataFrame(
    [
        (1, "Cen_01128", 0.8789375),
        (2, "Cen_01127", 0.865095),
        (3, "Cen_01126", 0.831432),
        (4, "Cen_01088", 0.81385),
        (5, "Cen_01087", 0.805875),
        (6, "Cen_01086", 0.800575),
        (7, "Cen_01125", 0.797818),
        (8, "Cen_01124", 0.791987),
        (9, "Cen_01085", 0.78525),
        (10, "Cen_00736", 0.7777),
    ],
    columns=["posicao", "cenario", "score"],
)

top10

## Bloco 2 - Tabela de referência (120 linhas)

Junta o Top 10 ao `kpis.parquet` (10 cenários x 12 anos) e grava `top10_referencia.csv` com os 22 KPIs das abas, na mesma ordem em que aparecem na planilha:

- **Aba 10 - Fluxo de caixa (método direto):** as 10 linhas `dfc_*`, de recebimentos a caixa final.
- **Aba 8 - Índices da DFC:** `taxa_conversao`, `fcl`, `fco_ebitda`, `fco_lucro_liquido`, `capex_receita`, `capex_da`, `cobertura_juros_caixa`, `payout_caixa` e `cobertura_dividendos`.
- **Aba 9 - Capital de giro:** `pmr`, `pmp` e `ciclo_financeiro` (PMP e ciclo ficam vazios no Ano 12, onde a conta de fornecedores não existe).

Os valores em R$ são divididos por 1 milhão e ganham o sufixo `_mi`. A última linha mostra o cenário em 1º lugar, com os anos nas colunas.

In [ ]:
kpis = pd.read_parquet(PASTA_STAGING / "kpis.parquet")

fluxo_de_caixa = [
    "dfc_recebimentos",
    "dfc_pagamentos",
    "dfc_fco",
    "dfc_investimentos",
    "dfc_juros_liquidos",
    "dfc_dividendos",
    "dfc_financiamento",
    "dfc_variacao_liquida",
    "dfc_caixa_inicial",
    "dfc_caixa_final",
]
indices_dfc = [
    "taxa_conversao",
    "fcl",
    "fco_ebitda",
    "fco_lucro_liquido",
    "capex_receita",
    "capex_da",
    "cobertura_juros_caixa",
    "payout_caixa",
    "cobertura_dividendos",
]
capital_de_giro = ["pmr", "pmp", "ciclo_financeiro"]

colunas_em_reais = fluxo_de_caixa + ["fcl"]
colunas_razao = [
    "taxa_conversao",
    "fco_ebitda",
    "fco_lucro_liquido",
    "capex_receita",
    "capex_da",
    "cobertura_juros_caixa",
    "payout_caixa",
    "cobertura_dividendos",
]

referencia = top10.merge(kpis, on="cenario").sort_values(["posicao", "ano_num"])

for coluna in colunas_em_reais:
    referencia[coluna + "_mi"] = (referencia[coluna] / 1e6).round(3)
for coluna in colunas_razao:
    referencia[coluna] = referencia[coluna].round(4)
for coluna in capital_de_giro:
    referencia[coluna] = referencia[coluna].round(2)

nomes_no_arquivo = []
for coluna in fluxo_de_caixa + indices_dfc + capital_de_giro:
    if coluna in colunas_em_reais:
        nomes_no_arquivo.append(coluna + "_mi")
    else:
        nomes_no_arquivo.append(coluna)

colunas_finais = ["posicao", "cenario", "score", "ano_calendario"] + nomes_no_arquivo
referencia = referencia[colunas_finais]

referencia.to_csv(PASTA_STAGING / "top10_referencia.csv", index=False)

print("Tabela de referência:", referencia.shape, "-> top10_referencia.csv")

primeiro = referencia[referencia["posicao"] == 1]
primeiro.set_index("ano_calendario").drop(columns=["posicao", "cenario", "score"]).T

## Bloco 3 - Métricas e justificativas do Top 10 (10 linhas)

Copia da aba `ranking_cenarios` da planilha de Contabilidade (células A8 a I17) as métricas e a justificativa de cada cenário do Top 10, junta ao score do bloco 1 e grava `top10_ranking.csv`, que alimenta a seção de ranking do dashboard.

In [ ]:
metricas = pd.DataFrame(
    [
        (
            1,
            "Cen_01128",
            100416.905,
            91133.973,
            47512.562,
            20524.096,
            1.4635,
            359.967,
            "Maior criação de valor dos 1.200 cenários (VP dos EVAs de R$ 20524,096 mi) e maior lucro acumulado. Único ponto fraco: aperto de liquidez em 2028 (1 ano-risco).",
        ),
        (
            2,
            "Cen_01127",
            99654.827,
            90401.359,
            47029.073,
            20369.689,
            1.4369,
            301.316,
            "Mesmo perfil do 1º e valor quase igual (R$ 20369,689 mi). Fica atrás pelo caixa mínimo menor: R$ 301,316 mi contra R$ 359,967 mi.",
        ),
        (
            3,
            "Cen_01126",
            98899.111,
            89674.864,
            46549.622,
            20216.555,
            1.4103,
            242.87,
            "Fecha a família líder: lucro e valor no topo, mas o caixa mais apertado dos três (mínimo de R$ 242,870 mi).",
        ),
        (
            4,
            "Cen_01088",
            93122.441,
            84009.252,
            43295.9,
            19187.337,
            1.9664,
            445.16,
            "O mais seguro da simulação: maior caixa mínimo (R$ 445,160 mi) e nenhum ano-risco. Cria menos valor que o pódio.",
        ),
        (
            5,
            "Cen_01087",
            92417.174,
            83342.918,
            42875.575,
            19052.332,
            1.9883,
            393.064,
            "Réplica do 4º, sem ano-risco, um degrau abaixo em valor (R$ 135,005 mi a menos) e em caixa.",
        ),
        (
            6,
            "Cen_01086",
            91717.73,
            82682.09,
            42458.755,
            18918.439,
            2.0197,
            341.151,
            "O mais líquido do trio conservador (LC média de 2,02), mas com a menor escala entre os seis primeiros.",
        ),
        (
            7,
            "Cen_01125",
            98149.705,
            88954.436,
            46074.176,
            20064.685,
            1.3837,
            184.628,
            "Cria mais valor que o 4º, 5º e 6º, mas cai pela liquidez: caixa mínimo de R$ 184,628 mi e aperto em 2028.",
        ),
        (
            8,
            "Cen_01124",
            97406.556,
            88240.026,
            45602.702,
            19914.067,
            1.3572,
            126.589,
            "Alto valor para o acionista, porém a menor folga de caixa do Top 10 (R$ 126,589 mi).",
        ),
        (
            9,
            "Cen_01085",
            91024.059,
            82026.721,
            42045.411,
            18785.647,
            2.0653,
            289.42,
            "Consistente: LC média de 2,07 e nenhum ano-risco, mas sem destaque em valor.",
        ),
        (
            10,
            "Cen_00736",
            87282.711,
            78730.1,
            40964.574,
            17934.142,
            1.8874,
            162.776,
            "Único fora das duas famílias dominantes; sem ano-risco, porém com VP dos EVAs só 1,0% acima da mediana.",
        ),
    ],
    columns=[
        "posicao",
        "cenario",
        "receita_liquida_acumulada_mi",
        "ebitda_acumulado_mi",
        "lucro_liquido_acumulado_mi",
        "vp_dos_evas_mi",
        "liquidez_corrente_media",
        "caixa_minimo_mi",
        "justificativa",
    ],
)

ranking = top10.merge(metricas, on=["posicao", "cenario"], validate="one_to_one")

ranking.to_csv(PASTA_STAGING / "top10_ranking.csv", index=False)

print("Ranking do Top 10:", ranking.shape, "-> top10_ranking.csv")

ranking.drop(columns="justificativa")